# 03 - Preparar Tarjetas y QC (listas para clasificar)

Ejecuta la preparación real de tarjetas usando `exploration/aidev/preparation/rejection_cards.py`.

Inputs/outputs (sobrescribibles por defecto):
- input: `exploration/aidev/sampling/outputs/merged_after_rework_sample.csv`
- output: `exploration/aidev/preparation/outputs/merged_after_rework_cards_seed_20260510.csv`
- summary: `exploration/aidev/preparation/outputs/merged_after_rework_cards_seed_20260510_summary.json`


In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PY = PROJECT_ROOT / '.venv' / 'bin' / 'python'
CARDS_SCRIPT = PROJECT_ROOT / 'exploration/aidev/preparation/rejection_cards.py'

IN_SAMPLE = PROJECT_ROOT / 'exploration/aidev/sampling/outputs/merged_after_rework_sample.csv'
OUT_CSV = PROJECT_ROOT / 'exploration/aidev/preparation/outputs/merged_after_rework_cards_seed_20260510.csv'
OUT_SUMMARY = PROJECT_ROOT / 'exploration/aidev/preparation/outputs/merged_after_rework_cards_seed_20260510_summary.json'

def run(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    print('[cmd]', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=True, text=True, capture_output=True)

if not IN_SAMPLE.exists():
    raise FileNotFoundError('[error] Falta IN_SAMPLE. Ejecuta antes 02_sampling_merged_after_rework.ipynb')

print('[info] IN_SAMPLE=', IN_SAMPLE)
print('[info] OUT_CSV=', OUT_CSV)
print('[info] OUT_SUMMARY=', OUT_SUMMARY)


In [ ]:
print('\n[step] Ejecutando preparación de tarjetas (consultando Parquet remotos)...')
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_SUMMARY.parent.mkdir(parents=True, exist_ok=True)
cmd = [
    str(PY), str(CARDS_SCRIPT),
    '--sample-csv', str(IN_SAMPLE),
    '--output-csv', str(OUT_CSV),
    '--summary-json', str(OUT_SUMMARY),
]
run(cmd)
print('[ok] preparación completada')


In [ ]:
print('\n[step] QC rápido desde summary.json')
summary = json.loads(OUT_SUMMARY.read_text(encoding='utf-8'))
print('[info] source_card_count=', summary.get('source_card_count'))
print('[info] card_count=', summary.get('card_count'))
print('[info] filtered_out_without_human_comments=', summary.get('filtered_out_without_human_comments'))
print('[info] evidence_source_counts=', summary.get('evidence_source_counts'))
print('[info] review_state_counts=', summary.get('review_state_counts'))
print('[info] agent_counts=', summary.get('agent_counts'))
row_count = sum(1 for _ in OUT_CSV.open('r', encoding='utf-8')) - 1
print('[info] csv_rows=', row_count)
if row_count != summary.get('card_count'):
    print('[warn] csv_rows != card_count')
else:
    print('[ok] conteo consistente')
